# Poromechanics

Learning goals:
1. Set up and run a 3d simulation with poromechanics
2. Import elliptic fractures from file
3. Use wells to control fluid injection
4. Use line search to stabilize the simulations


In [75]:
# The usual imports.
import numpy as np
import porepy as pp

from porepy.numerics.nonlinear import line_search
from porepy.applications.boundary_conditions.model_boundary_conditions import (
    HydrostaticBoundaryPressureValues,
    BoundaryConditionsMechanicsNeumann,
    LithostaticBoundaryStressValues,
)
from porepy.applications.initial_conditions.model_initial_conditions import (
    InitialConditionHydrostaticPressureValues,
)
from porepy.examples.geothermal_reservoir import WellBoundaryConditions
from porepy.viz.data_saving_model_mixin import FractureDeformationExporting

import logging

logger = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO)

DOMAIN_SIZE = 1000


class Geometry:
    def set_domain(self):

        self._domain = pp.Domain(
            {
                "xmin": 0,
                "xmax": DOMAIN_SIZE,
                "ymin": 0,
                "ymax": DOMAIN_SIZE,
                "zmin": 0,
                "zmax": DOMAIN_SIZE,
            }
        )

    def set_fractures(self):
        f_1 = pp.EllipticFracture(
            DOMAIN_SIZE * np.array([0.5, 0.5, 0.5]), DOMAIN_SIZE * 0.4, DOMAIN_SIZE * 0.3, 0, 0, 0
        )
        f_2 = pp.EllipticFracture(
            DOMAIN_SIZE * np.array([0.7, 0.5, 0.5]), DOMAIN_SIZE * 0.3, DOMAIN_SIZE * 0.2, 0, 0, 90
        )
        fractures = [f_1, f_2]
        self._fractures = fractures

In [76]:
class WellSpecification:
    """A mixin adding two wells to a 3d model.

    By default, one straight vertical well and one kinked well are added to a cubic
    domain. The domain size and well mesh size can be controlled by the parameters
    ``domain_sizes`` and ``well_mesh_size``, respectively.

    A sketch of the setup in the x-z plane is provided in the comments of the method
    :meth:`set_well_network`.
    """
    def set_well_network(self):
        """Set the well geometry"""

        well_1 = pp.Well(
            np.array([[0.3 * DOMAIN_SIZE, 0.3 * DOMAIN_SIZE], 
            [0.5 * DOMAIN_SIZE, 0.5 * DOMAIN_SIZE], [DOMAIN_SIZE, 0.2 * DOMAIN_SIZE]]),
            tags={"well_name": "injection_well"},
        )
        self._wells = [well_1]

        mesh_size = self.params.get("well_mesh_size", {"mesh_size": 0.1 * DOMAIN_SIZE})
        self.well_network = pp.WellNetwork3d(
            domain=self._domain, wells=self._wells, parameters=mesh_size
        )


In [77]:
class BoundaryConditions(
    HydrostaticBoundaryPressureValues,
    BoundaryConditionsMechanicsNeumann,
    LithostaticBoundaryStressValues,
):
    pass

In [78]:
# Set hydrostatic and lithostatic boundary conditions for the flow and mechanics problem
class SimulationSetup(
    Geometry,
    # Activate gravity.
    pp.constitutive_laws.GravityForce,
    # Set the fracture permeability through the cubic law.
    pp.constitutive_laws.CubicLawPermeability,
    WellSpecification,
    WellBoundaryConditions,
    BoundaryConditions,
    FractureDeformationExporting,
    InitialConditionHydrostaticPressureValues,
    pp.models.solution_strategy.ContactIndicators,
    pp.Poromechanics,
):
    def initialize_data_saving(self):
        super().initialize_data_saving()

        self.jump_history = {sd: [] for sd in self.mdg.subdomains(dim=self.nd-1)}
        self.jump_history_time = []


    def after_nonlinear_convergence(self):
        super().after_nonlinear_convergence()
        data = self.data_to_export()

        values = {sd: {} for sd in self.mdg.subdomains(dim=self.nd-1)}
        self.jump_history_time.append(self.time_manager.time)

        for sd, variable, value in data:
            if sd in values and variable == "displacement_jump":
                self.jump_history[sd].append(value)


In [79]:
# Define time schedule for the simulation.
schedule = np.array([0, pp.HOUR, 10 * pp.HOUR])

# Add initialization time interval.
dt_init = 30 * pp.DAY
schedule += dt_init * 12 
schedule = np.insert(schedule, 0, 0.0)

# Define injection pressures as list of len = schedule.size. For other protocol
# values, broadcasting of single values is used for simplicity. The following
# schedule is somewhat arbitrary, but meant to represent a ramping up of injection
# pressures over time. The initial low pressure represents a start from near
# hydrostatic conditions.

# We ramp up from 1e5 to 5e6 Pa during initialization (well is closed using a
# Neumann BC), then ramp up to 9e6 Pa at injection start (1 hour), then increase to
# 11e6 Pa after 10 hours, and finally to 15e6 Pa after 200 days.
injection_pressures = [1e5, 5e6, 9e6, 11e6, 15e6]  # [Pa]
# Convenient shortening of simulation schedule for quick simulations. The point is
# that injection_pressures must match the size of schedule.
schedule_length = schedule.size
schedule = schedule[:schedule_length]
injection_pressures = injection_pressures[:schedule_length]

time_manager = pp.TimeManager(
    schedule=schedule,
    dt_init=dt_init,
    constant_dt=False,
    dt_min_max=(0.1 * pp.MINUTE, max(pp.HOUR, dt_init)),
    iter_optimal_range=(6, 10),  # Allow more iterations than default.
    iter_relax_factors=(0.5, 1.8),  # More aggressive relaxation
)

In [80]:
schedule

array([       0., 31104000., 31107600., 31140000.])

In [81]:
solid_values = pp.solid_values.basalt
solid_values.update(
    {
        "dilation_angle": 0.1,  # [rad]
        # Uncomment next two lines to include elastic fracture deformation, aka
        # "Barton-Bandis" model for normal fracture deformation.
        # "fracture_normal_stiffness": 1.1e8,  # [Pa m^-1]
        # "maximum_elastic_fracture_opening": 1e-3,  # [m]
        "normal_permeability": 1.0e-10,  # [m^2]
        "residual_aperture": 1e-3,  # [m]
        "well_radius": 0.1,  # [m]
    }
)

In [82]:
model_params = {
    # Set time manager.
    "time_manager": time_manager,
    # Set physical parameters.
    "lithostatic_stress_multipliers": np.array([0.8, 1.4, 1.0]),
    "injection_well_pressures": injection_pressures,
    "production_well_pressures": pp.ATMOSPHERIC_PRESSURE,  # = 1.01325e5 Pa
    "material_constants": {
        "solid": pp.SolidConstants(**solid_values),  # type: ignore[arg-type]
        "fluid": pp.FluidComponent(**pp.fluid_values.water),  # type: ignore[arg-type]
        "numerical": pp.NumericalConstants(characteristic_displacement=1e-2),
    },
    "reference_variable_values": pp.ReferenceVariableValues(pressure=1e6),
    "units": pp.Units(m=1.0, kg=1.0e5, K=1.0),
    # Set geometry and meshing related parameters.
    "grid_type": "simplex",
    "meshing_arguments": {
        "cell_size": 0.7 * DOMAIN_SIZE,  # Base cell size for meshing.
        "cell_size_fracture": 0.3 * DOMAIN_SIZE,
        "cell_size_min": 0.1 * DOMAIN_SIZE,
    },
    "domain_sizes": 1.0,
    # Line search: Scale the indicator used for the local_line_search (see below)
    # adaptively to increase robustness.
    "adaptive_indicator_scaling": 1,
}

In [83]:
solver_params = {
    "prepare_simulation": True,
    "nl_max_iterations": 25,  # Max iterations of a nonlinear solver (Newton)
    "nl_convergence_inc_atol": 1e-7,  # Increment norm
    "nl_convergence_res_atol": 1e-7,  # Residual norm
    "nl_divergence_inc_atol": 1e12,
    "nl_divergence_res_atol": 1e12,
    # Line search / Solution Strategies. These are considered "advanced" options,
    # improving the robustness of the nonlinear solver at the cost of some
    # additional computational overhead. Delete/comment the following lines for the
    # default Newton's method.
    "nonlinear_solver": line_search.ConstraintLineSearchNonlinearSolver,
    # Set to 1 to use turn on a residual-based line search. This involves some extra
    # residual evaluations and may be quite costly.
    "global_line_search": 0,
    # Set to 0 to use turn off the tailored line search, see the class
    # ConstraintLineSearchNonlinearSolver. This line search is cheap and has proven
    # effective for (some versions of) this particular simulation setup.
    "local_line_search": 1,
}

In [ ]:
model = SimulationSetup(model_params)
pp.run_time_dependent_model(model, solver_params)

INFO:porepy.fracs.simplex:Grid creation completed. Elapsed time 0.027612924575805664
INFO:porepy.fracs.simplex:Created 1 3-d grids with 2165 cells
INFO:porepy.fracs.simplex:Created 2 2-d grids with 126 cells
INFO:porepy.fracs.simplex:Created 1 1-d grids with 4 cells


In [71]:
for sd, jump_history in model.jump_history.items():
    print(f"Fracture {sd.frac_num + 1}")
    for i, val in enumerate(jump_history):
        print(f"Time {model.jump_history_time[i]}: Max displacement jump = {model.units.convert_units(val.max(), 'm', to_si=True)} m")

Fracture 1


AttributeError: 'SimulationSetup' object has no attribute 'jump_history_time'